In [1]:
# Centralized imports (cleaned)
from bioio import BioImage
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tifffile import imwrite, imread
import tifffile
from skimage.segmentation import expand_labels, clear_border
from skimage.measure import regionprops_table
from cellpose import models
import napari
from liffile import LifFile
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import colorsys
from scipy.spatial import KDTree
from skimage.measure import regionprops
from skimage.transform import resize, rescale
from pathlib import Path
import pandas as pd
import scipy.ndimage as ndi
from skimage.measure import regionprops
from instanseg import InstanSeg
import time
from cellpose import models, utils as cellpose_utils
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_3d_segmentation

available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if available else 0
device_name = torch.cuda.get_device_name(0) if available and device_count > 0 else None

status = {
    "cuda_available": available,
    "device_count": device_count,
    "device_name": device_name,
}
print(status)

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\torch_em\util\image.py:6: UserWarning: elf has switched from the affogato, vigra, and nifty librares to https://github.com/computational-cell-analytics/bioimage-cpp as new backed for custom functionality implemented in C++, e.g. mutex watershed, multicut etc. This may lead to some changes in behavior and interface. If you run into issues with the new version consider installing a version < 0.9. Please also consider raising an issue on github so that we are aware of issues with the migration.
  from elf.io import open_file
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found.

{'cuda_available': True, 'device_count': 1, 'device_name': 'NVIDIA GeForce RTX 5080'}


In [2]:

_MODEL_CACHE = {} # cache loaded models across instances so we don't have to reload them for each image

class SegmentationComparisons:
    """Run several nuclear-segmentation methods (and parameter sweeps) on the same image.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X), where
    channel `dapi_channel` is DAPI and channel `bf_channel` is brightfield/phase.
    Every (method, parameter-combo) writes a label mask + overlay PNG to
    `output_dir` and contributes one row to the results table.
    """

    def __init__(self, input_csv, index, scale_factor_xy=3, scale_factor_z=2,
        custom_model_path=r"C:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish\cellpose_model",
        output_dir=Path(r"Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons"), dapi_channel=0, bf_channel=1):
        self.input_csv = input_csv
        self.index = index
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)
        self.custom_model_path = custom_model_path

        row = input_csv.loc[index]
        self.two_channel_image_path = Path(row["2_channel_tif_save_path"])
        self.dapi_channel = int(dapi_channel)
        self.bf_channel = int(bf_channel)
        self.image_name = row["image_name"]

        self.output_dir = Path(output_dir)
        self.existing_segmentation = tifffile.imread(Path(row["existing_segmentation_path"]))

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # label -> label array
        self.counts = {}   # label -> object count

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    def rescale_image(self):
        """Downsample DAPI and brightfield channels and compute z-anisotropy."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)
        bf = self.two_channel_image[:, self.bf_channel].astype(np.float32)

        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)
        self.bf_ds = rescale(bf, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)

        # downsample the existing/prior label mask with nearest-neighbour (order=0, no anti-aliasing)
        # so labels are preserved exactly and it stays pixel-aligned with the new segmentations
        self.existing_segmentation_ds = rescale(
            self.existing_segmentation, (z_ratio, xy_ratio, xy_ratio),
            order=0, anti_aliasing=False, preserve_range=True).astype(self.existing_segmentation.dtype)

        # channel-last 2-channel stack for cellpose-SAM: (Z, Y, X, 2) = [DAPI, BF]
        self.two_channel_ds = np.stack([self.dapi_ds, self.bf_ds], axis=-1)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f}")

    @staticmethod
    def _get_cellpose_model(pretrained=None):
        key = str(pretrained)
        if key not in _MODEL_CACHE:
            if pretrained is None:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True)  # built-in CPSAM
            else:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True, pretrained_model=str(pretrained))
        return _MODEL_CACHE[key]

    # ---- segmentation methods: each takes its swept params and RETURNS a label volume ----

    def cellpose_sam_2d_stitched(self, channels="dapi_bf", stitch_threshold=0.1,
                                 cellprob_threshold=0.0, flow_threshold=0.4, min_size=500):
        """Cellpose-SAM, per-z then stitched in 3D. `channels`: 'dapi_bf' or 'dapi'."""
        model = self._get_cellpose_model()
        if channels == "dapi_bf":
            img, channel_axis = self.two_channel_ds, 3
        else:  # DAPI only
            img, channel_axis = self.dapi_ds, None
        seg, _, _ = model.eval(img, channel_axis=channel_axis, z_axis=0, do_3D=False,
                               stitch_threshold=stitch_threshold, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, flow_threshold=flow_threshold,
                               min_size=min_size)
        return seg

    def cellpose_sam_true_3d(self, cellprob_threshold=0.0, min_size=500):
        """Cellpose-SAM, native 3D (flow_threshold is ignored by cellpose in do_3D mode)."""
        model = self._get_cellpose_model()
        seg, _, _ = model.eval(self.two_channel_ds, channel_axis=3, z_axis=0, do_3D=True,
                               anisotropy=self.anisotropy, cellprob_threshold=cellprob_threshold,
                               min_size=min_size)
        return seg

    def cellpose_custom_3d(self, cellprob_threshold=0.0, min_size=500, batch_size=128, resample=False):
        """Custom cellpose model on DAPI only, run as native 3D."""
        model = self._get_cellpose_model(self.custom_model_path)
        seg, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=True, anisotropy=self.anisotropy,
                               cellprob_threshold=cellprob_threshold, min_size=min_size,
                               batch_size=int(batch_size), resample=bool(resample))
        return seg

    def instanseg_2d_stitched(self, stitch_threshold=0.1, pixel_size_scale=1.0, target="nuclei"):
        """InstanSeg per-z (DAPI only) then stitched in 3D.

        Swept parameters:
        - `stitch_threshold`: IoU cutoff for linking 2D masks across z.
        - `pixel_size_scale`: multiplies the pixel size handed to InstanSeg, which controls
          the effective object scale the model assumes (smaller -> finer/more objects).
        - `target`: which InstanSeg output to keep ('nuclei' or 'cells').
        """
        if "instanseg" not in _MODEL_CACHE:
            _MODEL_CACHE["instanseg"] = InstanSeg("fluorescence_nuclei_and_cells", verbosity=0)
        model = _MODEL_CACHE["instanseg"]
        pixel_size = self.pixel_size_um_2d * float(pixel_size_scale)
        z_masks = []
        for z in range(self.dapi_ds.shape[0]):
            plane = self.dapi_ds[z][None]  # (C=1, H, W), DAPI only
            labeled, _ = model.eval_small_image(plane, pixel_size, target=target)
            lab = np.asarray(labeled.cpu() if hasattr(labeled, "cpu") else labeled).squeeze()
            if lab.ndim == 3:  # (n_outputs, H, W) -> take the first output
                lab = lab[0]
            z_masks.append(lab.astype(np.uint32))
        stacked = np.stack(z_masks, axis=0)
        return cellpose_utils.stitch3D(stacked, stitch_threshold=stitch_threshold)

    def micro_sam_3d(self, model_type="vit_b_lm"):
        """micro-sam automatic 3D segmentation on DAPI only."""
        key = f"microsam_{model_type}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = get_predictor_and_segmenter(model_type=model_type)
        predictor, segmenter = _MODEL_CACHE[key]
        return automatic_3d_segmentation(self.dapi_ds.astype(np.float32), predictor, segmenter)

    # ---- output helpers ----

    @staticmethod
    def _param_label(params):
        """Short filesystem-safe string describing a parameter combo."""
        parts = [f"{k}={v}" for k, v in params.items()]
        label = "__".join(parts) if parts else "default"
        return label.replace(".", "p").replace("-", "neg")

    def _save_overlay_png(self, seg, png_path, method_name, suptitle):
        """Save a max-projection figure with five panels:
        DAPI, brightfield, the NEW method labels over brightfield, the
        EXISTING (prior) segmentation over brightfield, and a diff panel
        (green = new only, red = old only).
        """
        dapi_mip = self.dapi_ds.max(axis=0)
        bf_mip = self.bf_ds.max(axis=0)
        new_label_mip = seg.max(axis=0) if seg.ndim == 3 else seg

        # existing/prior segmentation, downsampled to match the new segmentations (pixel-aligned)
        existing = self.existing_segmentation_ds
        existing_mip = existing.max(axis=0) if existing.ndim == 3 else existing

        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        axes[0].imshow(dapi_mip, cmap="gray")
        axes[0].set_title("DAPI (max projection)")
        axes[1].imshow(bf_mip, cmap="gray")
        axes[1].set_title("Brightfield (max projection)")

        axes[2].imshow(bf_mip, cmap="gray")
        new_overlay = np.ma.masked_where(new_label_mip == 0, new_label_mip)
        axes[2].imshow(new_overlay, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
        axes[2].set_title(f"NEW method: {method_name}")

        axes[3].imshow(bf_mip, cmap="gray")
        existing_overlay = np.ma.masked_where(existing_mip == 0, existing_mip)
        axes[3].imshow(existing_overlay, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
        axes[3].set_title("EXISTING (prior) segmentation")

       # Diff panel: align BOTH masks to bf_mip.shape (the canonical reference) before
        # comparing, since rescale rounding / micro-sam output can be ±1 pixel vs bf_mip.
        ref_shape = bf_mip.shape

        def _align(arr):
            if arr.shape == ref_shape:
                return arr
            return resize(arr, ref_shape, order=0, anti_aliasing=False,
                          preserve_range=True).astype(arr.dtype)

        new_binary = _align(new_label_mip) > 0
        existing_binary = _align(existing_mip) > 0
        new_not_old = new_binary & ~existing_binary
        old_not_new = existing_binary & ~new_binary
        bf_norm = bf_mip.astype(np.float32)
        bf_norm = (bf_norm - bf_norm.min()) / (bf_norm.max() - bf_norm.min() + 1e-8)
        diff_rgb = np.stack([bf_norm, bf_norm, bf_norm], axis=-1)
        diff_rgb[new_not_old] = [0.0, 1.0, 0.0]  # green: new but not old
        diff_rgb[old_not_new] = [1.0, 0.0, 0.0]  # red: old but not new

        axes[4].imshow(diff_rgb, interpolation="nearest")
        axes[4].set_title("Diff: green=new only, red=old only")

        for ax in axes:
            ax.axis("off")
        fig.suptitle(suptitle)
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    def _build_jobs(self, cpsam2d_channels, cpsam2d_stitch, cellprob_thresholds,
                    cellprob_3d, flow_threshold, microsam_models, min_size,
                    instanseg_stitch, instanseg_pixel_size_scales, instanseg_targets, include):
        """Expand the parameter grids into a flat list of (method, params, fn) jobs.

        `cellprob_thresholds` is swept only for the cheap 2D-stitched method; the
        expensive native-3D methods run once at the single value `cellprob_3d`
        (each cellprob value would otherwise re-run the full ~45 min 3D network)."""
        jobs = []
        if "cpsam_2dstitch" in include:
            for ch in cpsam2d_channels:
                for stitch in cpsam2d_stitch:
                    for cp in cellprob_thresholds:
                        jobs.append(("cpsam_2dstitch",
                                     {"channels": ch, "stitch_threshold": stitch,
                                      "cellprob_threshold": cp, "flow_threshold": flow_threshold,
                                      "min_size": min_size},
                                     self.cellpose_sam_2d_stitched))
        if "cpsam_true3d" in include:
            jobs.append(("cpsam_true3d",
                         {"cellprob_threshold": cellprob_3d, "min_size": min_size},
                         self.cellpose_sam_true_3d))
        if "cellpose_custom_3d" in include:
            jobs.append(("cellpose_custom_3d",
                         {"cellprob_threshold": cellprob_3d, "min_size": min_size},
                         self.cellpose_custom_3d))
        if "instanseg" in include:
            for stitch in instanseg_stitch:
                for ps in instanseg_pixel_size_scales:
                    for tgt in instanseg_targets:
                        jobs.append(("instanseg",
                                     {"stitch_threshold": stitch, "pixel_size_scale": ps, "target": tgt},
                                     self.instanseg_2d_stitched))
        if "microsam" in include:
            for mt in microsam_models:
                jobs.append(("microsam", {"model_type": mt}, self.micro_sam_3d))
        return jobs

    ######### MAIN PART############
    def run_sweep(self,
                  cpsam2d_channels=("dapi_bf", "dapi"),
                  cpsam2d_stitch=(0.1, 0.25),
                  cellprob_thresholds=(0.0, 1.0),
                  cellprob_3d=0.0,
                  flow_threshold=0.4,
                  microsam_models=("vit_b_lm", "vit_l_lm"),
                  min_size=500,
                  instanseg_stitch=(0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75),
                  instanseg_pixel_size_scales=(1.0,),
                  instanseg_targets=("nuclei", "cells"),
                  include=("cpsam_true3d", "cellpose_custom_3d", "instanseg", "microsam")):
        """Run every (method, parameter-combo). Saves a TIF + overlay PNG per combo and
        returns one results row per successful run.

        `cellprob_thresholds` is swept for the cheap 2D-stitched method only; the
        expensive native-3D methods run once at `cellprob_3d`. The cpsam 2D-stitch
        method is left available but excluded from the default `include`. InstanSeg
        is swept over `instanseg_stitch` x `instanseg_pixel_size_scales` x `instanseg_targets`."""
        self.rescale_image()
        jobs = self._build_jobs(cpsam2d_channels, cpsam2d_stitch, cellprob_thresholds,
                                cellprob_3d, flow_threshold, microsam_models, min_size,
                                instanseg_stitch, instanseg_pixel_size_scales, instanseg_targets, include)
        print(f"{self.image_name}: {len(jobs)} jobs queued")

        self.run_results = []
        for method_name, params, fn in jobs:
            label = f"{method_name}__{self._param_label(params)}"
            try:
                t0 = time.time()
                seg = fn(**params)
                time_taken = time.time() - t0

                count = int(np.unique(seg).size - (1 if (seg == 0).any() else 0))
                self.counts[label] = count
                self.results[label] = seg
                print(f"  {label}: {count} objects, {time_taken:.1f}s")

                method_dir = self.output_dir / method_name
                method_dir.mkdir(parents=True, exist_ok=True)
                stem = f"{self.image_name}__{label}"
                imwrite(method_dir / f"{stem}.tif", seg.astype(np.uint32))
                self._save_overlay_png(seg, method_dir / f"{stem}.png", method_name, stem)

                row = {"image_name": self.image_name, "method": method_name,
                       "time_taken": time_taken, "objects_found": count}
                row.update(params)
                self.run_results.append(row)
            except Exception as exc:
                print(f"[SKIP] {label}: {type(exc).__name__}: {exc}")

        return self.run_results


In [3]:
input_csv = pd.read_excel(r"z:\Bel\Jorge_SPACEFISH_Examples\image_locations.xlsx")
input_csv.head()


,image_name,path,dapi_channel,bf_channel,image_type,scene_id,"censor region (z1,z2,x1_z1, x2_z1, y1_z1,y2_z1,x1_z2,x2_z2,y1_z2,y2_z2)",existing_segmentation_path,2_channel_tif_save_path
0,dev4_1_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
1,dev4_2_6h,Z:\Jorge\20241125_repeats_6h_2d_pin255\2024112...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
2,dev4_3_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,2,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
3,dev7_3_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
4,dev7_2_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,1,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...


In [ ]:
# run the full parameter sweep on every image, collecting one results row per successful run
# save a per-image CSV as soon as each image finishes so an early termination doesn't lose data
all_results = []
for index in reversed(input_csv.index):
    comparison = SegmentationComparisons(input_csv, index=index, scale_factor_xy=3, scale_factor_z=2)
    image_results = comparison.run_sweep()
    all_results.extend(image_results)

    # write this image's results immediately
    image_df = pd.DataFrame(image_results)
    safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(comparison.image_name))
    image_csv_path = comparison.output_dir / f"results_{safe_name}.csv"
    image_df.to_csv(image_csv_path, index=False)
    print(f"saved {len(image_df)} rows -> {image_csv_path}")

# single combined CSV across all images, methods, and parameter combos
results_df = pd.DataFrame(all_results)
results_csv_path = comparison.output_dir / "segmentation_comparison_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"saved {len(results_df)} rows -> {results_csv_path}")
results_df

# to trim the sweep, pass narrower grids, e.g.:
# comparison.run_sweep(cpsam2d_channels=("dapi",), microsam_models=("vit_b_lm",))
# to run a single image:
# comparison = SegmentationComparisons(input_csv, index=0)
# comparison.run_sweep()

downsampled DAPI shape (48, 683, 683), anisotropy 2.818
dev1_2_P1: 20 jobs queued


INFO:cellpose.core:** TORCH CUDA version installed and working. **
INFO:cellpose.core:>>>> using GPU (CUDA)
INFO:cellpose.models:>>>> loading model C:\Users\taylorhearn\.cellpose\models\cpsam
INFO:cellpose.models:resizing 3D image with anisotropy=2.8178465300340614
INFO:cellpose.core:running YX: 135 planes of size (683, 683)
INFO:cellpose.core:100%|##########| 135/135 [00:48<00:00,  2.80it/s]
INFO:cellpose.core:running ZY: 683 planes of size (135, 683)
INFO:cellpose.core:100%|##########| 342/342 [01:00<00:00,  5.67it/s]
INFO:cellpose.core:running ZX: 683 planes of size (135, 683)
INFO:cellpose.core:100%|##########| 342/342 [01:01<00:00,  5.52it/s]
INFO:cellpose.models:network run in 175.19s
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\cellpose\dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur perf

  cpsam_true3d__cellprob_threshold=0p0__min_size=500: 60 objects, 182.2s


INFO:cellpose.core:** TORCH CUDA version installed and working. **
INFO:cellpose.core:>>>> using GPU (CUDA)
INFO:cellpose.models:>>>> loading model C:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish\cellpose_model
INFO:cellpose.models:resizing 3D image with anisotropy=2.8178465300340614
INFO:cellpose.core:running YX: 135 planes of size (683, 683)
INFO:cellpose.core:100%|##########| 17/17 [09:47<00:00, 34.55s/it]
INFO:cellpose.core:running ZY: 683 planes of size (135, 683)
INFO:cellpose.core:100%|##########| 22/22 [12:15<00:00, 33.44s/it]
INFO:cellpose.core:running ZX: 683 planes of size (135, 683)
INFO:cellpose.core:100%|##########| 22/22 [12:18<00:00, 33.57s/it]
INFO:cellpose.models:network run in 2066.60s
INFO:cellpose.models:masks created in 2.76s


  cellpose_custom_3d__cellprob_threshold=0p0__min_size=500: 56 objects, 2077.8s


...

Merge segmentation: 100%|██████████| 1/1 [00:07<00:00,  7.65s/it]


  microsam__model_type=vit_l_lm: 1148 objects, 35.8s
saved 20 rows -> Z:\Bel\Jorge_SPACEFISH_Examples\cellpose_comparisons\results_dev9_4_day4.csv
downsampled DAPI shape (63, 1303, 1311), anisotropy 2.813
dev10_2_day4: 20 jobs queued


INFO:cellpose.models:resizing 3D image with anisotropy=2.8128084700288
INFO:cellpose.core:running YX: 177 planes of size (1303, 1311)
INFO:cellpose.core:100%|##########| 177/177 [03:15<00:00,  1.11s/it]
INFO:cellpose.core:running ZY: 1303 planes of size (177, 1311)
INFO:cellpose.core:100%|##########| 1303/1303 [03:31<00:00,  6.15it/s]
INFO:cellpose.core:running ZX: 1311 planes of size (177, 1303)
INFO:cellpose.core:100%|##########| 1311/1311 [03:50<00:00,  5.70it/s]
INFO:cellpose.models:network run in 653.89s
INFO:cellpose.models:masks created in 4.36s


  cpsam_true3d__cellprob_threshold=0p0__min_size=500: 257 objects, 679.4s


INFO:cellpose.models:resizing 3D image with anisotropy=2.8128084700288
INFO:cellpose.core:running YX: 177 planes of size (1303, 1311)
INFO:cellpose.core:100%|##########| 89/89 [11:59<00:00,  8.08s/it]
INFO:cellpose.core:running ZY: 1303 planes of size (177, 1311)
INFO:cellpose.core:100%|##########| 73/73 [18:01<00:00, 14.81s/it]
INFO:cellpose.core:running ZX: 1311 planes of size (177, 1303)
INFO:cellpose.core:16%|#6        | 12/73 [03:05<15:43, 15.46s/it]
